---

## Section 9 — CNN: ResNet-50 (Transfer Learning)

**Requires GPU — run in Google Colab.**

### Two-phase fine-tuning

| Phase | Layers trained | Epochs | LR |
|---|---|---|---|
| A | Classification head only (frozen backbone) | 5 | 1e-3 |
| B | Head + `layer4` unfrozen | 10 | 1e-4 |

Phase A trains quickly and finds a good starting point for the head. Phase B fine-tunes the top ResNet block with a lower learning rate to avoid destroying ImageNet features.

---

In [ ]:
# ── 8.1  Build ResNet-50 ──────────────────────────────────────────────────────
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Freeze all layers
for p in resnet50.parameters():
    p.requires_grad = False

# Replace classification head → 101 classes
resnet50.fc = nn.Linear(2048, 101)
resnet50    = resnet50.to(DEVICE)

frozen  = sum(p.numel() for p in resnet50.parameters() if not p.requires_grad)
trained = sum(p.numel() for p in resnet50.parameters() if p.requires_grad)
print(f"ResNet-50 — frozen: {frozen:,}  trainable: {trained:,}")

assert resnet50(torch.randn(2, 3, 224, 224).to(DEVICE)).shape == (2, 101)
print("Forward pass: OK → shape (2, 101)")

# ── 8.2  Phase A — head only ──────────────────────────────────────────────────
opt_rn = Adam(resnet50.fc.parameters(), lr=1e-3)
sch_rn = ReduceLROnPlateau(opt_rn, factor=0.5, patience=2)
hist_rn = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

print("\n── Phase A (frozen backbone, head only) ──")
for epoch in range(1, 6):
    tr_loss, tr_acc = train_one_epoch(resnet50, img_train_loader, criterion_cnn, opt_rn)
    vl_loss, vl_acc = evaluate(resnet50, img_val_loader, criterion_cnn)
    sch_rn.step(vl_loss)
    hist_rn["train_loss"].append(tr_loss); hist_rn["val_loss"].append(vl_loss)
    hist_rn["train_acc"].append(tr_acc);  hist_rn["val_acc"].append(vl_acc)
    torch.save({"epoch": epoch, "model_state": resnet50.state_dict(), "val_loss": vl_loss},
               WEIGHTS / f"cnn_resnet50_ckpt_ep{epoch:02d}.pt")
    print(f"  Ep {epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {vl_loss:.4f}/{vl_acc:.3f}")

# ── 8.3  Phase B — unfreeze layer4 ───────────────────────────────────────────
print("\n── Phase B (layer4 + head unfrozen) ──")
for p in resnet50.layer4.parameters():
    p.requires_grad = True

opt_rn_b = Adam(filter(lambda p: p.requires_grad, resnet50.parameters()), lr=1e-4)
sch_rn_b = ReduceLROnPlateau(opt_rn_b, factor=0.5, patience=2)
best_rn_loss = float("inf")

for epoch in range(6, 16):
    tr_loss, tr_acc = train_one_epoch(resnet50, img_train_loader, criterion_cnn, opt_rn_b)
    vl_loss, vl_acc = evaluate(resnet50, img_val_loader, criterion_cnn)
    sch_rn_b.step(vl_loss)
    hist_rn["train_loss"].append(tr_loss); hist_rn["val_loss"].append(vl_loss)
    hist_rn["train_acc"].append(tr_acc);  hist_rn["val_acc"].append(vl_acc)
    torch.save({"epoch": epoch, "model_state": resnet50.state_dict(), "val_loss": vl_loss},
               WEIGHTS / f"cnn_resnet50_ckpt_ep{epoch:02d}.pt")
    print(f"  Ep {epoch} | train {tr_loss:.4f}/{tr_acc:.3f} | val {vl_loss:.4f}/{vl_acc:.3f}")
    if vl_loss < best_rn_loss:
        best_rn_loss = vl_loss
        torch.save(resnet50.state_dict(), WEIGHTS / "cnn_resnet50.pt")

# ── 8.4  Learning curves ──────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_rn["train_loss"], label="train"); ax1.plot(hist_rn["val_loss"], label="val")
ax1.axvline(4.5, color="gray", linestyle=":", label="Phase A→B")
ax1.set_title("ResNet-50 — Loss"); ax1.legend()
ax2.plot(hist_rn["train_acc"],  label="train"); ax2.plot(hist_rn["val_acc"],  label="val")
ax2.axvline(4.5, color="gray", linestyle=":")
ax2.set_title("ResNet-50 — Accuracy"); ax2.legend()
plt.tight_layout()
plt.savefig(FIGURES / "cnn_resnet50_curves.png", dpi=100)
plt.show()

# ── 8.5  Final test evaluation ────────────────────────────────────────────────
resnet50.load_state_dict(torch.load(WEIGHTS / "cnn_resnet50.pt"))
_, rn_test_acc = evaluate(resnet50, img_test_loader, criterion_cnn)

rn_preds, rn_labels = [], []
resnet50.eval()
with torch.no_grad():
    for imgs, labels in img_test_loader:
        preds = resnet50(imgs.to(DEVICE)).argmax(1).cpu()
        rn_preds.extend(preds.tolist()); rn_labels.extend(labels.tolist())

rn_f1 = f1_score(rn_labels, rn_preds, average="macro")
print(f"\nResNet-50  — Test accuracy: {rn_test_acc:.4f}  Macro F1: {rn_f1:.4f}")
print(f"CNN Scratch — Test accuracy: {test_acc:.4f}  Macro F1: {macro_f1:.4f}")
print(f"\n✓ Section 8 complete — weights saved to {WEIGHTS / 'cnn_resnet50.pt'}")

NameError: name 'models' is not defined

---

## Section 10 — CNN Explainability: Grad-CAM

Grad-CAM visualises which spatial regions of a food photo drove the CNN's classification decision. We show 6 examples: 2 correct predictions, 2 wrong predictions, and 2 where top-1 is wrong but the correct class is in top-5.

---

In [ ]:
# ── 9  Grad-CAM explainability ────────────────────────────────────────────────
from torchcam.methods import GradCAM
from torchcam.utils  import overlay_mask
from torchvision.transforms.functional import to_pil_image

resnet50.load_state_dict(torch.load(WEIGHTS / "cnn_resnet50.pt", map_location=DEVICE))
resnet50.eval()

cam_extractor = GradCAM(resnet50, target_layer="layer4")

# Collect 6 examples: 2 correct, 2 wrong top-1, 2 top-5 correct
examples = {"correct": [], "wrong": [], "top5_correct": []}
for imgs, labels in img_test_loader:
    imgs, labels = imgs.to(DEVICE), labels
    logits = resnet50(imgs)
    top5   = logits.topk(5, dim=1).indices.cpu()
    top1   = logits.argmax(1).cpu()
    for i in range(len(labels)):
        lbl = labels[i].item()
        if top1[i] == lbl and len(examples["correct"]) < 2:
            examples["correct"].append((imgs[i], lbl, top1[i].item()))
        elif top1[i] != lbl and lbl not in top5[i].tolist() and len(examples["wrong"]) < 2:
            examples["wrong"].append((imgs[i], lbl, top1[i].item()))
        elif top1[i] != lbl and lbl in top5[i].tolist() and len(examples["top5_correct"]) < 2:
            examples["top5_correct"].append((imgs[i], lbl, top1[i].item()))
    if all(len(v) >= 2 for v in examples.values()):
        break

all_examples = (
    [("correct",      *e) for e in examples["correct"]] +
    [("wrong",        *e) for e in examples["wrong"]] +
    [("top-5 hit",    *e) for e in examples["top5_correct"]]
)

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
class_names = ds_train.classes

for col, (case, img_t, true_lbl, pred_lbl) in enumerate(all_examples):
    logits = resnet50(img_t.unsqueeze(0))
    cam    = cam_extractor(pred_lbl, logits)[0]

    # Original image (un-normalise)
    orig = img_t.cpu() * torch.tensor(IMAGENET_STD).view(3,1,1) \
               + torch.tensor(IMAGENET_MEAN).view(3,1,1)
    orig = orig.clamp(0, 1)

    overlay = overlay_mask(to_pil_image(orig), to_pil_image(cam.squeeze(0)), alpha=0.5)

    axes[0, col].imshow(to_pil_image(orig))
    axes[0, col].set_title(f"True: {class_names[true_lbl]}\nPred: {class_names[pred_lbl]}\n[{case}]",
                            fontsize=7)
    axes[0, col].axis("off")

    axes[1, col].imshow(overlay)
    axes[1, col].set_title("Grad-CAM", fontsize=7)
    axes[1, col].axis("off")

plt.suptitle("Grad-CAM — ResNet-50 on Food-101 test images", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "gradcam.png", dpi=100)
plt.show()
cam_extractor.remove_hooks()
print("✓ Section 10 complete.")

---

## Section 11 — LSTM Baseline

Unidirectional LSTM trained on CPU. Serves as the baseline for comparison with BiLSTM in Section 12.

**Architecture:** GloVe embedding (frozen) → LSTM(hidden=128) → Linear(128→15)

Random baseline for 15 classes = 6.7% accuracy.

---

In [ ]:

# ── 10  LSTM Baseline ─────────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, embed_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embed_matrix, dtype=torch.float32), requires_grad=False
        )
        self.lstm       = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        _, (h, _) = self.lstm(emb)
        return self.classifier(self.dropout(h.squeeze(0)))

lstm_model = LSTMClassifier(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=128,
    n_classes=len(GRAPE_CLASSES), embed_matrix=embedding_matrix
).to(DEVICE)

print(f"LSTM params: {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")

# Move CLASS_WEIGHTS to DEVICE here — it was kept on CPU in Section 6 to avoid
# a CUDA health-check failure at weight computation time
criterion_txt = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
opt_lstm      = Adam(filter(lambda p: p.requires_grad, lstm_model.parameters()), lr=1e-3)
hist_lstm     = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def train_txt_epoch(model, loader, criterion, optimizer):
    model.train(); total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        n          += len(yb)
    return total_loss / n, correct / n

@torch.no_grad()
def eval_txt(model, loader, criterion):
    model.eval(); total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        total_loss += criterion(logits, yb).item() * len(yb)
        correct    += (logits.argmax(1) == yb).sum().item()
        n          += len(yb)
    return total_loss / n, correct / n

best_lstm_loss = float("inf")

for epoch in range(1, 4):
    tr_l, tr_a = train_txt_epoch(lstm_model, txt_train_loader, criterion_txt, opt_lstm)
    vl_l, vl_a = eval_txt(lstm_model, txt_val_loader, criterion_txt)
    hist_lstm["train_loss"].append(tr_l); hist_lstm["val_loss"].append(vl_l)
    hist_lstm["train_acc"].append(tr_a);  hist_lstm["val_acc"].append(vl_a)
    print(f"Epoch {epoch} | train {tr_l:.4f}/{tr_a:.3f} | val {vl_l:.4f}/{vl_a:.3f}")

    # Save checkpoint every epoch
    torch.save({"epoch": epoch, "model_state": lstm_model.state_dict(), "val_loss": vl_l},
               WEIGHTS / f"lstm_ckpt_ep{epoch:02d}.pt")
    # Keep the best model
    if vl_l < best_lstm_loss:
        best_lstm_loss = vl_l
        torch.save(lstm_model.state_dict(), WEIGHTS / "lstm.pt")

# Test evaluation
_, lstm_test_acc = eval_txt(lstm_model, txt_test_loader, criterion_txt)
lstm_preds, lstm_lbls = [], []
lstm_model.eval()
with torch.no_grad():
    for xb, yb in txt_test_loader:
        lstm_preds.extend(lstm_model(xb.to(DEVICE)).argmax(1).cpu().tolist())
        lstm_lbls.extend(yb.tolist())

lstm_f1 = f1_score(lstm_lbls, lstm_preds, average="macro")
print(f"\nLSTM — Test accuracy: {lstm_test_acc:.4f}  Macro F1: {lstm_f1:.4f}")
print(f"(Random baseline: {1/len(GRAPE_CLASSES):.4f})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
a1.plot(hist_lstm["train_loss"], label="train"); a1.plot(hist_lstm["val_loss"], label="val")
a1.set_title("LSTM — Loss"); a1.legend()
a2.plot(hist_lstm["train_acc"],  label="train"); a2.plot(hist_lstm["val_acc"],  label="val")
a2.set_title("LSTM — Accuracy"); a2.legend()
plt.tight_layout(); plt.savefig(FIGURES / "lstm_curves.png", dpi=100); plt.show()
print(f"✓ Section 11 complete — lstm.pt saved to {WEIGHTS / 'lstm.pt'}")

---

## Section 12 — BiLSTM with Attention

### Dual role

The BiLSTM serves two purposes in this project:

1. **Classifier** — predicts grape variety from a tasting note (15 classes, trained here)
2. **Encoder** — at inference (Section 14), finds the most representative Vivino review per grape by comparing each review's hidden-state vector against the grape centroid

Sub-section 12.3 builds the grape centroids and implements the review retrieval function.

**Architecture:** GloVe embedding → BiLSTM(hidden=128, bidirectional=True, output=256) → attention → Linear(256→15)

---

In [ ]:
# ── 12.1  Architecture ────────────────────────────────────────────────────────
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes, embed_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embed_matrix, dtype=torch.float32), requires_grad=False
        )
        self.bilstm  = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn_w  = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_dim * 2, n_classes)

    def encode(self, x):
        """Return attention-weighted sentence vector [B, 2*hidden]."""
        emb, _ = self.bilstm(self.dropout(self.embedding(x)))
        scores  = self.attn_w(emb).squeeze(-1)           # [B, T]
        weights = torch.softmax(scores, dim=1).unsqueeze(2)  # [B, T, 1]
        return (emb * weights).sum(dim=1), weights        # [B, 256], [B, T, 1]

    def forward(self, x):
        ctx, _ = self.encode(x)
        return self.classifier(self.dropout(ctx))

bilstm_model = BiLSTMAttention(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=128,
    n_classes=len(GRAPE_CLASSES), embed_matrix=embedding_matrix
).to(DEVICE)

print(f"BiLSTM params: {sum(p.numel() for p in bilstm_model.parameters() if p.requires_grad):,}")

# ── 12.2  Train BiLSTM ────────────────────────────────────────────────────────
opt_bi   = Adam(filter(lambda p: p.requires_grad, bilstm_model.parameters()), lr=1e-3)
hist_bi  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_bi_loss = float("inf")

for epoch in range(1, 6):
    tr_l, tr_a = train_txt_epoch(bilstm_model, txt_train_loader, criterion_txt, opt_bi)
    vl_l, vl_a = eval_txt(bilstm_model, txt_val_loader, criterion_txt)
    hist_bi["train_loss"].append(tr_l); hist_bi["val_loss"].append(vl_l)
    hist_bi["train_acc"].append(tr_a);  hist_bi["val_acc"].append(vl_a)
    torch.save({"epoch": epoch, "model_state": bilstm_model.state_dict(), "val_loss": vl_l},
               WEIGHTS / f"bilstm_ckpt_ep{epoch:02d}.pt")
    print(f"Epoch {epoch} | train {tr_l:.4f}/{tr_a:.3f} | val {vl_l:.4f}/{vl_a:.3f}")
    if vl_l < best_bi_loss:
        best_bi_loss = vl_l
        torch.save(bilstm_model.state_dict(), WEIGHTS / "bilstm.pt")

# Test evaluation
_, bi_test_acc = eval_txt(bilstm_model, txt_test_loader, criterion_txt)
bi_preds, bi_lbls = [], []
bilstm_model.eval()
with torch.no_grad():
    for xb, yb in txt_test_loader:
        bi_preds.extend(bilstm_model(xb.to(DEVICE)).argmax(1).cpu().tolist())
        bi_lbls.extend(yb.tolist())

bi_f1 = f1_score(bi_lbls, bi_preds, average="macro")
print(f"\nBiLSTM — Test accuracy: {bi_test_acc:.4f}  Macro F1: {bi_f1:.4f}")
print(f"LSTM   — Test accuracy: {lstm_test_acc:.4f}  Macro F1: {lstm_f1:.4f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
a1.plot(hist_bi["train_loss"], label="BiLSTM train"); a1.plot(hist_bi["val_loss"], label="BiLSTM val")
a1.plot(hist_lstm["train_loss"], "--", label="LSTM train"); a1.plot(hist_lstm["val_loss"], "--", label="LSTM val")
a1.set_title("Loss comparison"); a1.legend(fontsize=7)
a2.plot(hist_bi["train_acc"], label="BiLSTM train"); a2.plot(hist_bi["val_acc"], label="BiLSTM val")
a2.plot(hist_lstm["train_acc"], "--", label="LSTM train"); a2.plot(hist_lstm["val_acc"], "--", label="LSTM val")
a2.set_title("Accuracy comparison"); a2.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIGURES / "bilstm_vs_lstm.png", dpi=100); plt.show()

# ── 12.3  BiLSTM as encoder: grape centroids + review retrieval ───────────────
bilstm_model.load_state_dict(torch.load(WEIGHTS / "bilstm.pt", map_location=DEVICE))
bilstm_model.eval()

grape_centroids = {}
with torch.no_grad():
    for grape_idx, grape in enumerate(GRAPE_CLASSES):
        indices = [i for i, l in enumerate(lbl_train) if l == grape_idx]
        if not indices: continue
        vecs = []
        for i in indices:
            xb = torch.tensor(X_train[i], dtype=torch.long).unsqueeze(0).to(DEVICE)
            ctx, _ = bilstm_model.encode(xb)
            vecs.append(ctx.squeeze(0).cpu().numpy())
        grape_centroids[grape] = np.mean(vecs, axis=0)

np.save(str(WEIGHTS / "grape_centroids.npy"), grape_centroids)

def get_representative_review(grape, df, top_n_candidates=500):
    """Return (review_text, wine_label, rating_pct) for the review
    whose BiLSTM vector is closest to the grape centroid."""
    centroid = grape_centroids[grape]
    candidates = df[df["grape_class"] == grape].nlargest(top_n_candidates, "rating_pct")
    best_score, best_row = -1, None
    bilstm_model.eval()
    with torch.no_grad():
        for _, row in candidates.iterrows():
            tokens = encode_and_pad(tokenise(row["review_text"]), MAX_SEQ_LEN)
            xb = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(DEVICE)
            ctx, _ = bilstm_model.encode(xb)
            vec   = ctx.squeeze(0).cpu().numpy()
            score = np.dot(vec, centroid) / (np.linalg.norm(vec) * np.linalg.norm(centroid) + 1e-9)
            if score > best_score:
                best_score, best_row = score, row
    return best_row["review_text"], best_row["wine_label"], int(best_row["rating_pct"])

print("Testing review retrieval for Cabernet Sauvignon ...")
rev, wine, pct = get_representative_review("Cabernet Sauvignon", df_wine_mapped)
print(f"  Wine   : {wine}")
print(f"  Rating : {pct}%")
print(f"  Review : {rev[:120]} ...")
print(f"\n✓ Section 12 complete — bilstm.pt and grape_centroids.npy saved.")

---

## Section 13 — BiLSTM Explainability: Attention Weights

Renders 6 tasting notes (one per grape variety, sampled from test set) with words colour-highlighted proportionally to their attention weight. High-weight words are the tokens that drove the grape variety prediction.

---

In [ ]:
# ── 13  Attention weight visualisation ───────────────────────────────────────
import matplotlib.colors as mcolors

bilstm_model.load_state_dict(torch.load(WEIGHTS / "bilstm.pt", map_location=DEVICE))
bilstm_model.eval()
IDX_TO_GRAPE = {i: g for g, i in GRAPE_TO_IDX.items()}

# One review per grape, first 6 grapes for readability
fig, axes = plt.subplots(6, 1, figsize=(18, 14))
fig.suptitle("BiLSTM attention weights — which words drove the grape prediction", fontsize=12)

for ax, grape_idx in enumerate(range(6)):
    grape = GRAPE_CLASSES[grape_idx]
    # Pick a correctly-classified test sample for this grape
    sample_idx = next(
        (i for i, (l, p) in enumerate(zip(bi_lbls, bi_preds)) if l == grape_idx and p == l), None
    )
    if sample_idx is None:
        ax.axis("off"); continue

    tokens_ids = X_test[sample_idx]
    xb = torch.tensor(tokens_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, weights = bilstm_model.encode(xb)
    weights_np = weights.squeeze().cpu().numpy()

    # Recover word strings (up to actual length, skip PAD)
    IDX_TO_WORD = {v: k for k, v in VOCAB.items()}
    words = [IDX_TO_WORD.get(t, "<UNK>") for t in tokens_ids if t != 0]
    attn  = weights_np[:len(words)]
    attn  = (attn - attn.min()) / (attn.max() - attn.min() + 1e-9)

    # Render coloured text
    cmap = plt.get_cmap("YlOrRd")
    x_pos = 0.0
    for word, weight in zip(words[:30], attn[:30]):  # show first 30 tokens
        color = cmap(weight)
        ax.text(x_pos, 0.5, word + " ", transform=ax.transAxes,
                fontsize=9, color="black", backgroundcolor=color,
                verticalalignment="center")
        x_pos += len(word) * 0.012 + 0.01
        if x_pos > 0.95: break

    ax.set_title(f"{grape} — pred: {IDX_TO_GRAPE[bi_preds[sample_idx]]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES / "bilstm_attention.png", dpi=100)
plt.show()
print("✓ Section 13 complete.")

---

## ⏸ ON PAUSE — Section 14: Joint Food-Wine Compatibility Classifier

This section requires Word2Vec grape embeddings (Section 6.7) to generate compatibility labels for food-image × wine-review pairs. It will be completed once the Word2Vec pairing system is resumed.

**Depends on:** Section 3.3 (flavor table) + Section 6.6–6.7 (Word2Vec + grape embeddings)

---

## Section 14 — Joint Model: Food-Wine Compatibility Classifier (+10 pts)

**Requires GPU — run in Google Colab.**

Pairs are built from **training data only** to prevent leakage into test evaluation.

**Architecture:** frozen CNN encoder (ResNet-50, 2048-d) + frozen BiLSTM encoder (256-d) → concat (2304-d) → FC head → sigmoid

Only the FC head is trained.

---

---

## Model Comparison Summary

Required by the assignment rubric (Section 5.6): a single chart comparing CNN, RNN, and joint model performance.

---

In [ ]:
# ── Model comparison chart ────────────────────────────────────────────────────
# Joint model AUC-ROC on held-out pairs (from pair_val_ds)
joint_model.load_state_dict(torch.load(WEIGHTS / "joint_model.pt", map_location=DEVICE))
joint_model.eval()

jt_preds_prob, jt_lbls = [], []
with torch.no_grad():
    for imgs, texts, lbls in pair_val_loader:
        imgs, texts = imgs.to(DEVICE), texts.to(DEVICE)
        jt_preds_prob.extend(joint_model(imgs, texts).cpu().tolist())
        jt_lbls.extend(lbls.tolist())

jt_auc = roc_auc_score(jt_lbls, jt_preds_prob)
jt_acc = sum((p > 0.5) == l for p, l in zip(jt_preds_prob, jt_lbls)) / len(jt_lbls)

# ── Summary table ──────────────────────────────────────────────────────────────
print(f"{'Model':<30} {'Task':<35} {'Test Acc':>9}  {'Macro F1':>9}  {'AUC-ROC':>9}")
print("─" * 95)
print(f"{'CNN Scratch':<30} {'Food classification (101 cls)':<35} {test_acc:>9.4f}  {macro_f1:>9.4f}  {'—':>9}")
print(f"{'CNN ResNet-50':<30} {'Food classification (101 cls)':<35} {rn_test_acc:>9.4f}  {rn_f1:>9.4f}  {'—':>9}")
print(f"{'LSTM baseline':<30} {'Grape variety (15 cls)':<35} {lstm_test_acc:>9.4f}  {lstm_f1:>9.4f}  {'—':>9}")
print(f"{'BiLSTM + attention':<30} {'Grape variety (15 cls)':<35} {bi_test_acc:>9.4f}  {bi_f1:>9.4f}  {'—':>9}")
print(f"{'Joint model':<30} {'Food-wine compatibility (0/1)':<35} {'—':>9}  {'—':>9}  {jt_auc:>9.4f}")

# ── Comparison bar chart ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CNN comparison
ax = axes[0]
ax.bar(["Scratch", "ResNet-50"], [test_acc, rn_test_acc], color=["#4C72B0", "#DD8452"])
ax.set_ylim(0, 1); ax.set_title("CNN — Test Accuracy\n(Food-101, 101 classes)")
ax.set_ylabel("Accuracy")
for i, v in enumerate([test_acc, rn_test_acc]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

# RNN comparison
ax = axes[1]
ax.bar(["LSTM", "BiLSTM+Attn"], [lstm_test_acc, bi_test_acc], color=["#4C72B0", "#DD8452"])
ax.set_ylim(0, 1); ax.set_title("RNN — Test Accuracy\n(WineSensed, 15 grape classes)")
ax.set_ylabel("Accuracy")
for i, v in enumerate([lstm_test_acc, bi_test_acc]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

# Joint model
ax = axes[2]
ax.bar(["Joint Model"], [jt_auc], color=["#55A868"])
ax.set_ylim(0, 1); ax.set_title("Joint Model — AUC-ROC\n(Food-wine compatibility, binary)")
ax.set_ylabel("AUC-ROC")
ax.text(0, jt_auc + 0.01, f"{jt_auc:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES / "model_comparison.png", dpi=100)
plt.show()
print("✓ Model comparison complete.")

---

## ⏸ ON PAUSE — Section 15: Business Integration & Recommendation Pipeline

This section assembles the full Wine Peer pipeline end-to-end: CNN food label → Word2Vec flavor similarity → grape selection → BiLSTM review retrieval → recommendation card. It requires the Word2Vec pairing system (Sections 3.3 + 6.6–6.7) to be completed first.

**Depends on:** Section 3.3 (flavor table) + Section 6.6–6.7 (Word2Vec + grape embeddings)

---

## Section 15 — Business Integration: Recommendation Card and 20-Example Table

Assembles the full Wine Peer pipeline end-to-end and runs it on 20 test images.

---

In [ ]:
# ── 15  Recommendation pipeline ───────────────────────────────────────────────

# Load all models
resnet50.load_state_dict(torch.load(WEIGHTS / "cnn_resnet50.pt", map_location=DEVICE))
bilstm_model.load_state_dict(torch.load(WEIGHTS / "bilstm.pt",   map_location=DEVICE))
joint_model.load_state_dict(torch.load(WEIGHTS / "joint_model.pt", map_location=DEVICE))
resnet50.eval(); bilstm_model.eval(); joint_model.eval()

grape_emb_arr = np.load(str(WEIGHTS / "grape_embeddings.npy"), allow_pickle=True).item()

def cosine_top1(vec, embeddings):
    best, best_grape = -1, GRAPE_CLASSES[0]
    for grape, emb in embeddings.items():
        s = np.dot(vec, emb) / (np.linalg.norm(vec) * np.linalg.norm(emb) + 1e-9)
        if s > best: best, best_grape = s, grape
    return best_grape

def recommend(img_pil):
    # 1. CNN → food label
    img_t  = val_test_transform(img_pil).unsqueeze(0).to(DEVICE)
    logits = resnet50(img_t)
    top3   = logits.topk(3, dim=1)
    food_label  = ds_train.classes[top3.indices[0, 0].item()]
    confidence  = torch.softmax(logits, dim=1)[0, top3.indices[0, 0]].item()

    # 2. Food flavor table → keyword sets
    profile = FOOD_FLAVOR_TABLE.get(food_label, {
        "complement": "rich savory earthy",
        "contrast":   "crisp acidic mineral",
        "balance":    "light fruity clean",
    })

    # 3. Word2Vec → grape per intent
    recs = {}
    for intent in ["complement", "contrast", "balance"]:
        vec = embed_keywords(profile[intent], w2v_model)
        recs[intent] = cosine_top1(vec, grape_emb_arr)

    # 4. Joint model → compatibility score per recommendation
    joint_scores = {}
    for intent, grape in recs.items():
        rev_sample = df_wine_mapped[df_wine_mapped["grape_class"] == grape]["review_text"].iloc[0]
        tokens = torch.tensor(encode_and_pad(tokenise(rev_sample), MAX_SEQ_LEN), dtype=torch.long) \
                        .unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            joint_scores[intent] = joint_model(img_t, tokens).item()

    # 5. BiLSTM → representative review + wine + rating
    card = {}
    for intent, grape in recs.items():
        rev_text, wine_name, rating = get_representative_review(grape, df_wine_mapped)
        card[intent] = {
            "grape":       grape,
            "wine":        wine_name,
            "rating_pct":  rating,
            "review":      rev_text,
            "joint_score": joint_scores[intent],
        }

    return food_label, confidence, card

# ── 20-example table ──────────────────────────────────────────────────────────
rows = []
sample_indices = random.sample(range(len(img_test_ds)), 20)

for i in sample_indices:
    img_t, true_lbl = img_test_ds[i]
    img_np = img_t.cpu() * torch.tensor(IMAGENET_STD).view(3,1,1) \
                  + torch.tensor(IMAGENET_MEAN).view(3,1,1)
    img_pil = to_pil_image(img_np.clamp(0, 1))

    food, conf, card = recommend(img_pil)
    rows.append({
        "True food":       ds_train.classes[true_lbl],
        "CNN prediction":  food,
        "Confidence":      f"{conf:.2%}",
        "Complement grape": card["complement"]["grape"],
        "Complement wine":  card["complement"]["wine"][:30],
        "Complement %":     f"{card['complement']['rating_pct']}%",
        "Contrast grape":   card["contrast"]["grape"],
        "Contrast wine":    card["contrast"]["wine"][:30],
        "Contrast %":       f"{card['contrast']['rating_pct']}%",
        "Balance grape":    card["balance"]["grape"],
        "Balance wine":     card["balance"]["wine"][:30],
        "Balance %":        f"{card['balance']['rating_pct']}%",
        "Joint (compl)":    f"{card['complement']['joint_score']:.3f}",
    })

results_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 30)
display(results_df)
pd.reset_option("display.max_colwidth")

print("\n✓ Section 15 complete.")

---

## Section 16 — Business Framing, Ethics, and Team Contributions

### 16.1 Business framing

**What decision does this pipeline support?**
Wine Peer supports the moment-of-ordering decision: which wine to choose when a guest has already decided on a dish but has no wine knowledge. The app bridges the gap between food and wine without requiring the user to understand wine vocabulary.

**Who is the end user?**
- Restaurant guests ordering at the table (primary)
- Home cooks pairing wine with a meal they are preparing
- Wine shop staff recommending a bottle for a customer's stated dinner plan

**What is the cost of an error?**

| Error type | What it means | Cost |
|---|---|---|
| CNN wrong food label | Wrong flavor profile used | Recommendation is for the wrong dish |
| Word2Vec selects wrong grape | Pairing intent not met | Mildly inappropriate recommendation |
| BiLSTM retrieves wrong review | Review shown does not match the wine | User mis-impression of the wine's character |
| Joint model wrong score | Compatible pair scored low | User misses a good pairing |

---

### 16.2 Ethics and bias

| Bias | Source | Effect | Mitigation |
|---|---|---|---|
| **Geographic bias** | WineSensed skews toward European wines | Non-European regions under-represented | Expand dataset |
| **Language bias** | English reviews dominate | Flavor language skews Anglo-American | Include multilingual reviews |
| **Cultural food bias** | Food-101 dominated by Western dishes | Non-Western cuisines have poor CNN accuracy | Extend Food-101 |
| **Affluence bias** | Highly-rated wines are typically expensive | Recommendations skew premium | Add price-tier filtering |

---

### 16.3 Team contributions

| Section | Component | Team member |
|---|---|---|
| 1, 2, 3 | Environment setup, data loading, cleaning | |
| 4 | EDA — images and text | |
| 5, 7, 8, 9 | Image preprocessing, CNN scratch, ResNet-50, Grad-CAM | |
| 6, 11, 12, 13 | Text preprocessing, LSTM, BiLSTM, attention | |
| 14 | Joint model | |
| 15, 16 | Recommendation card, business framing, ethics | |
| Deployment | Streamlit app, HF Spaces | |

---

### References

- Bossard et al. (2014). *Food-101 — Mining Discriminative Components with Random Forests.* ECCV.
- Dakhoo et al. (2023). *WineSensed: a large-scale wine dataset.* NeurIPS 2023.
- Mikolov et al. (2013). *Efficient Estimation of Word Representations in Vector Space.* ICLR.
- Pennington et al. (2014). *GloVe: Global Vectors for Word Representation.* EMNLP.
- He et al. (2016). *Deep Residual Learning for Image Recognition.* CVPR.
- Selvaraju et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks.* ICCV.